# Study 906 — EM Local Bonds FX-Hedged 🌏

**EM local-currency bonds pay a fat local rate — but does stripping the FX leave a real,
harvestable carry, or does the boring USD-EM ETF still win?**

Emerging-market **local-currency** government bonds (EMLC, LEMB, EBND) yield far more than
their US-dollar cousins because they pay the *local* short rate (6–13 % in Brazil, Mexico,
Indonesia, South Africa). The catch: your USD return is `local_bond + EM_FX`, and the
currency leg is so volatile — and, over 2010–2026, so *negative* as the dollar rose — that
the carry disappears. The obvious fix: **hedge the FX**. No clean FX-hedged EM-local ETF
exists on US tape, so we use a **proxy**: a long **UUP** (dollar-index) overlay that gains
when the broad dollar rallies and EM currencies fall together.

*Numbers below are the frozen headline (`docs/results.md`, fingerprint `32605dae0c9c`);
the live cells run the fast synthetic control. **Proxy caveat:** UUP tracks the
developed-market DXY basket, not the EMLC currency basket — it strips only part of the EM-FX.*


## 1. The idea in one picture

An EM-local bond fund earns the local rate **plus** whatever the currency does in dollars. When the dollar is strong, EM currencies weaken together and drown the carry. A **long dollar-index overlay** (UUP) rises exactly then — so adding it back approximately cancels the currency drag, leaving (mostly) the local rate.

In [1]:
R = {'start': '2010-08-31', 'end': '2026-06-30', 'n': 191, 'fingerprint': '32605dae0c9c', 'emlc_uup_beta': -1.12, 't_emlc_uup': -11.91, 'emlc_uup_r2': 0.5, 'hedge_b': -1.118, 'unhedged_exc': 0.33, 'unhedged_sharpe': 0.03, 't_unhedged': 0.14, 'hedged_exc': 1.62, 'hedged_sharpe': 0.2, 't_hedged': 0.94, 'emb_exc': 3.1, 'emb_sharpe': 0.34, 't_emb': 1.48, 'prem_diff': -1.48, 't_prem_diff': -0.81, 'welch': -0.49, 'lemb_sharpe': 0.23, 'lemb_t': 0.99, 'lemb_prem': -1.03, 'ebnd_sharpe': 0.26, 'ebnd_t': 1.16, 'ebnd_prem': -1.33, 'boot_unh_lo': -0.38, 'boot_unh_hi': 0.47, 'boot_hed_lo': -0.21, 'boot_hed_hi': 0.69, 'boot_hed_fracneg': 0.17, 'boot_emb_lo': -0.05, 'boot_emb_hi': 0.91, 'wf_exc': 1.6, 'wf_sharpe': 0.2, 'wf_t': 0.79, 'wf_resid_fx': 0.1, 'era_early_hed': 1.76, 'era_early_t': 0.7, 'era_early_prem': -3.58, 'era_early_prem_t': -1.87, 'era_early_n': 125, 'era_late_hed': 1.34, 'era_late_t': 0.84, 'era_late_prem': 2.5, 'era_late_prem_t': 0.71, 'era_late_n': 66, 'dd_emlc': -32.3, 'dd_emb': -28.7, 'dd_hedged': -17.1, 'cost_charge': 0.46, 'cost_gross': 1.62, 'cost_net': 1.15, 'cost_net_t': 0.67, 'cost_net_sharpe': 0.15, 'cost_net_prem': -1.95, 'cost_net_prem_t': -1.07, 'planted_exc': 5.53, 'planted_t': 3.25, 'planted_b': -1.1, 'null_t_mean': -0.01, 'null_t_sd': 1.41, 'null_fire': 1, 'null_seeds': 20}

In [2]:
print(f"raw EMLC over cash : {R['unhedged_exc']:+.2f}%/yr  Sharpe {R['unhedged_sharpe']:+.2f}  (the FX drowns it)")
print(f"FX-stripped (hedged): {R['hedged_exc']:+.2f}%/yr  Sharpe {R['hedged_sharpe']:+.2f}  (the local carry surfaces)")
print(f"but plain USD-EM EMB: {R['emb_exc']:+.2f}%/yr  Sharpe {R['emb_sharpe']:+.2f}  (the boring sibling still wins)")

raw EMLC over cash : +0.33%/yr  Sharpe +0.03  (the FX drowns it)
FX-stripped (hedged): +1.62%/yr  Sharpe +0.20  (the local carry surfaces)
but plain USD-EM EMB: +3.10%/yr  Sharpe +0.34  (the boring sibling still wins)


## 2. Is the machinery honest? A live synthetic control

We plant a known local-rate carry in a seeded toy world (`carry>0`) under a dollar-explained FX drag, and check the overlay recovers it — and stays *silent* on the null (`carry=0`, FX present but no carry). No network.

In [3]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from em_hedged import data, strategy as st
planted = st.synthetic_detect(data.synthetic_world(carry_annual=0.04, seed=906))
null    = st.synthetic_detect(data.synthetic_world(carry_annual=0.0,  seed=906))
print('planted carry world: hedged HAC t = %+.2f  (should light up)' % planted['t_hedged'])
print('null (no carry)    : hedged HAC t = %+.2f  (should be ~0)'    % null['t_hedged'])

planted carry world: hedged HAC t = +3.25  (should light up)
null (no carry)    : hedged HAC t = +0.90  (should be ~0)


## 3. Where the hedge genuinely helps — the drawdown

Stripping the dollar move roughly **halves** the drawdown (EMLC **-32%** → hedged **-17%**). That is a real, mechanical diversification win — you sleep better. It just doesn't turn into a significant *return* premium.

In [4]:
print(f"EMLC unhedged max drawdown : {R['dd_emlc']:.1f}%")
print(f"hedged-EMLC max drawdown   : {R['dd_hedged']:.1f}%   <- the FX-strip cuts the pain")
print(f"EMB (USD-EM) max drawdown  : {R['dd_emb']:.1f}%")

EMLC unhedged max drawdown : -32.3%
hedged-EMLC max drawdown   : -17.1%   <- the FX-strip cuts the pain
EMB (USD-EM) max drawdown  : -28.7%


## 4. The honest verdict — a real mechanism, no bankable edge

The FX-strip is **mechanically real** (EMLC is ~50 % dollar-basket FX; hedging lifts the Sharpe +0.03 → +0.20 and halves the drawdown). **But the leftover local carry isn't a robust premium** — **+1.62 %/yr at HAC *t* = +0.94**, a bootstrap Sharpe CI of [-0.21, +0.69] that straddles zero — and it **loses to just owning USD-EM debt** (hedged − EMB = **-1.48 %/yr**). After the overlay's cost the gap only widens. **Signal: Weak. Tradability: Mirage** — the simpler EMB wins outright.